In [1]:
import cv2
import numpy as np
import math

from mindx.sdk import Tensor
from mindx.sdk import base

import IPython.display
import ipywidgets as widgets

# ========================= 前处理 =========================
def preprocess(img_bgr):
    img = cv2.resize(img_bgr, (256, 256))
    img = img.astype(np.float32)

    mean = np.array([123.675, 116.28, 103.53], dtype=np.float32)
    std = np.array([58.395, 57.12, 57.375], dtype=np.float32)

    img = (img - mean) / std
    img = img[:, :, ::-1].transpose((2, 0, 1))
    img = np.expand_dims(img, 0)

    img = np.ascontiguousarray(img, dtype=np.float32)
    img = Tensor(img)

    return img


# ========================= 推理 =========================
def infer(model, input_tensor):
    output_feat = model.infer([input_tensor])[0]
    return output_feat


# ========================= 手势分类 =========================
def classify_gesture(pts):

    if len(pts) < 21:
        return "UNKNOWN"

    # 关键点
    wrist      = pts[0]

    thumb_tip  = pts[4]
    index_tip  = pts[8]
    mid_tip    = pts[12]
    ring_tip   = pts[16]
    pinky_tip  = pts[20]

    thumb_ip   = pts[3]

    index_pip  = pts[6]
    mid_pip    = pts[10]
    ring_pip   = pts[14]
    pinky_pip  = pts[18]

    # ========================= 判断手指是否伸直 =========================
    def is_finger_straight(tip, pip):

        d_tip = np.hypot(
            tip[0] - wrist[0],
            tip[1] - wrist[1]
        )

        d_pip = np.hypot(
            pip[0] - wrist[0],
            pip[1] - wrist[1]
        )

        return d_tip > d_pip

    thumb_straight = is_finger_straight(thumb_tip, thumb_ip)

    index_straight = is_finger_straight(index_tip, index_pip)
    mid_straight   = is_finger_straight(mid_tip, mid_pip)
    ring_straight  = is_finger_straight(ring_tip, ring_pip)
    pinky_straight = is_finger_straight(pinky_tip, pinky_pip)

    # ============================================================
    # 1. GOOD 点赞
    # ============================================================
    if (thumb_straight and
        not index_straight and
        not mid_straight and
        not ring_straight and
        not pinky_straight):

        return "GOOD"

    # ============================================================
    # 2. OK 手势
    # 拇指和食指接近
    # 中指、无名指、小指伸直
    # ============================================================

    thumb_index_dist = np.hypot(
        thumb_tip[0] - index_tip[0],
        thumb_tip[1] - index_tip[1]
    )

    # 手掌尺度（自适应）
    palm_size = np.hypot(
        mid_tip[0] - wrist[0],
        mid_tip[1] - wrist[1]
    )

    if (thumb_index_dist < palm_size * 0.35 and
        mid_straight and
        ring_straight and
        pinky_straight):

        return "OK"

    # ============================================================
    # 3. YEAH 比耶
    # ============================================================
    if (index_straight and
        mid_straight and
        not ring_straight and
        not pinky_straight):

        return "YEAH"

    # ============================================================
    # 4. PALM 张开手掌
    # ============================================================
    if (index_straight and
        mid_straight and
        ring_straight and
        pinky_straight):

        return "PALM"

    # ============================================================
    # 5. FIST 握拳
    # ============================================================
    if (not index_straight and
        not mid_straight and
        not ring_straight and
        not pinky_straight):

        return "FIST"

    return "UNKNOWN"


# ========================= 后处理 =========================
def postprocess(img_bgr, output_feat):

    img_width = img_bgr.shape[1]
    img_height = img_bgr.shape[0]

    output_feat.to_host()

    output_feat_array = np.array(output_feat)[0]

    heatmaps_reshaped = output_feat_array.reshape((21, -1))

    idx = np.argmax(heatmaps_reshaped, 1).reshape((21, 1))
    maxvals = np.amax(heatmaps_reshaped, 1).reshape((21, 1))

    preds = np.tile(idx, (1, 2)).astype(np.float32)

    preds[:, 0] = preds[:, 0] % 64
    preds[:, 1] = preds[:, 1] // 64

    preds = np.where(
        np.tile(maxvals, (1, 2)) > 0.0,
        preds,
        -1
    )

    # 亚像素优化
    for k in range(21):

        heatmap = output_feat_array[k]

        px = int(preds[k][0])
        py = int(preds[k][1])

        if 1 < px < 63 and 1 < py < 63:

            diff = np.array([
                heatmap[py][px + 1] - heatmap[py][px - 1],
                heatmap[py + 1][px] - heatmap[py - 1][px]
            ])

            preds[k] += np.sign(diff) * 0.25

    all_preds = np.zeros((21, 3), dtype=np.float32)

    all_preds[:, 0:2] = preds
    all_preds[:, 2] = maxvals[:, 0]

    # ========================= 坐标映射回原图 =========================

    scale_x = img_width / 64.0
    scale_y = img_height / 64.0

    cx = img_width / 2
    cy = img_height / 2

    all_preds[:, 0] = all_preds[:, 0] * scale_x + (cx - img_width * 0.5)
    all_preds[:, 1] = all_preds[:, 1] * scale_y + (cy - img_height * 0.5)

    # ========================= 绘制关键点 =========================

    x_list = []
    y_list = []
    
    for p in all_preds:

        x, y, s = p

        if s > 0.3:

            x_list.append(x)
            y_list.append(y)

            cv2.circle(
                img_bgr,
                (int(x), int(y)),
                4,
                (255, 50, 60),
                -1
            )

    # ========================= 绘制骨架 =========================

    skeleton = [
        (0,1),(1,2),(2,3),(3,4),
        (0,5),(5,6),(6,7),(7,8),
        (0,9),(9,10),(10,11),(11,12),
        (0,13),(13,14),(14,15),(15,16),
        (0,17),(17,18),(18,19),(19,20)
    ]

    for connection in skeleton:

        start_idx, end_idx = connection

        x1, y1, s1 = all_preds[start_idx]
        x2, y2, s2 = all_preds[end_idx]

        # 两个点都检测到才画线
        if s1 > 0.3 and s2 > 0.3:

            cv2.line(
                img_bgr,
                (int(x1), int(y1)),
                (int(x2), int(y2)),
                (0, 255, 0),
                2
            )

    # ========================= 外接框 + 手势文字 =========================

    if len(x_list) > 10:

        x1 = int(min(x_list)) - 15
        y1 = int(min(y_list)) - 15

        x2 = int(max(x_list)) + 15
        y2 = int(max(y_list)) + 15

        cv2.rectangle(
            img_bgr,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        gesture = classify_gesture(all_preds)

        cv2.putText(
            img_bgr,
            f"Gesture: {gesture}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2
        )

    return img_bgr


# ========================= 图像转 bytes =========================
def img2bytes(image):
    return bytes(cv2.imencode('.jpg', image)[1])


# ========================= 初始化 =========================
base.mx_init()

DEVICE_ID = 0

model_path = 'landmark.om'

model = base.model(
    modelPath=model_path,
    deviceId=DEVICE_ID
)

# ========================= 打开摄像头 =========================
cap = cv2.VideoCapture(0)

image_widget = widgets.Image(
    format='jpeg',
    width=1024,
    height=768
)

display(image_widget)

# ========================= 主循环 =========================
try:

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        # 前处理
        img = preprocess(frame)

        # 推理
        res = infer(model, img)

        # 后处理
        frame = postprocess(frame, res)

        # 显示
        image_widget.value = img2bytes(frame)

except KeyboardInterrupt:

    cap.release()

finally:

    cap.release()

Begin to initialize Log.
The output directory of logs file exist.
Save logs information to specified directory.


I20260514 23:07:23.384234 88452 FileUtils.cpp:331] The input file is empty
I20260514 23:07:23.384317 88452 FileUtils.cpp:475] Check Other group permission: Current permission is 4, but required no greater than 0.


Image(value=b'', format='jpeg', height='768', width='1024')